## **RuleFit ( Rules from a Tree Ensemble + a Sparse Linear Model )**

**RuleFit** ( Friedman and Popescu, 2008 ) keeps the accuracy of a tree **ensemble** and the readability of **rules**:

1. A **gradient-boosting** ensemble of shallow trees is fitted; **every node** of every tree defines a candidate rule ( the conjunction of the conditions on its path )
2. Each rule becomes a **binary feature**; the original attributes are added as **linear terms**
3. A **sparse ( L1 ) logistic regression** selects the few rules and terms that matter, and gives each one a **weight**

The model is a **weighted sum of readable rules**: `score = intercept + w1·[rule 1] + w2·[rule 2] + … + w·attribute`. This notebook implements it with scikit-learn and measures the **sparsity, the contribution of the rules and of the linear terms ( ablation ), the regularisation path and the stability**.

**Notebooks of this series:** 1. CN2 Rule Induction · 2. Expert Systems · 3. Case-Based Reasoning · 4. Apriori for Classification ( CBA ) · 5. Simple Rule Learners ( ZeroR, OneR, PRISM ) · 6. RIPPER · 7. Rules from Decision Trees · 8. RuleFit · 9. Fuzzy Rule-Based Classifier

### **0. Environment Setup**

In [1]:
#Import Libs
import sklearn
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from cycler import cycler

In [2]:
#Define Colors Palette and Standards for Illustrations
RED, DARK, GRAY, LIGHT, SOFT = "#CD2027", "#404040", "#7F7F7F", "#D9D9D9", "#E9A0A3"

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

plt.rcParams.update({
    "figure.figsize": (7.5, 4.2),
    "figure.dpi": 100,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.grid": True,
    "grid.alpha": 0.25,
    "axes.titleweight": "bold",
    "axes.prop_cycle": cycler(color=[RED, DARK, SOFT, GRAY]),
})
pd.set_option("display.precision", 3)
pd.set_option("display.width", 120)

In [3]:
#Helper Functions: Evaluation Reports ( Confusion Matrix, Metrics, ROC and Precision-Recall )
from sklearn.metrics import (accuracy_score, balanced_accuracy_score, precision_score, recall_score, f1_score,
                             roc_auc_score, average_precision_score, log_loss, matthews_corrcoef,
                             confusion_matrix, roc_curve, precision_recall_curve)

def plot_confusion(cm, labels, ax, title="Confusion matrix"):
    ax.imshow(cm, cmap="Reds"); ax.grid(False)
    ax.set_xticks(range(len(labels))); ax.set_xticklabels(labels, rotation=45 if len(labels) > 4 else 0, ha="right" if len(labels) > 4 else "center")
    ax.set_yticks(range(len(labels))); ax.set_yticklabels(labels)
    for i in range(cm.shape[0]):
        for j in range(cm.shape[1]):
            ax.text(j, i, cm[i, j], ha="center", va="center", fontsize=10 if len(labels) > 4 else 13,
                    color="white" if cm[i, j] > cm.max() / 2 else DARK)
    ax.set_xlabel("predicted"); ax.set_ylabel("true"); ax.set_title(title)

def report_binary(y_true, y_pred, y_score, name="model", labels=("negative", "positive"), plot=True):
    """Confusion matrix, ROC and Precision-Recall curves + a Series with the main metrics (positive class = 1)."""
    y_true, y_pred, y_score = np.asarray(y_true), np.asarray(y_pred), np.asarray(y_score)
    metrics = {"accuracy": accuracy_score(y_true, y_pred), "balanced acc.": balanced_accuracy_score(y_true, y_pred),
               "precision": precision_score(y_true, y_pred, zero_division=0), "recall": recall_score(y_true, y_pred),
               "F1": f1_score(y_true, y_pred), "MCC": matthews_corrcoef(y_true, y_pred),
               "ROC AUC": roc_auc_score(y_true, y_score), "PR AUC": average_precision_score(y_true, y_score)}
    if plot:
        fig, axes = plt.subplots(1, 3, figsize=(15, 4.2))
        plot_confusion(confusion_matrix(y_true, y_pred), list(labels), axes[0], f"Confusion matrix – {name}")
        fpr, tpr, _ = roc_curve(y_true, y_score)
        axes[1].plot(fpr, tpr, color=RED, lw=2.4, label=f"AUC = {metrics['ROC AUC']:.3f}"); axes[1].plot([0, 1], [0, 1], color=GRAY, ls="--")
        axes[1].set_xlabel("false positive rate"); axes[1].set_ylabel("true positive rate"); axes[1].set_title("ROC curve"); axes[1].legend(loc="lower right")
        p, r, _ = precision_recall_curve(y_true, y_score)
        axes[2].plot(r, p, color=DARK, lw=2.4, label=f"AP = {metrics['PR AUC']:.3f}"); axes[2].axhline(y_true.mean(), color=GRAY, ls="--")
        axes[2].set_xlabel("recall"); axes[2].set_ylabel("precision"); axes[2].set_title("Precision–Recall curve"); axes[2].legend(loc="lower left")
        plt.tight_layout(); plt.show()
    return pd.Series(metrics, name=name)

def report_multiclass(y_true, y_pred, y_proba, labels, name="model", plot=True):
    """Confusion matrix, one-vs-rest ROC curves and per-class F1 + a Series with the main metrics."""
    y_true, y_pred = np.asarray(y_true), np.asarray(y_pred)
    k = len(labels)
    metrics = {"accuracy": accuracy_score(y_true, y_pred), "balanced acc.": balanced_accuracy_score(y_true, y_pred),
               "macro precision": precision_score(y_true, y_pred, average="macro", zero_division=0),
               "macro recall": recall_score(y_true, y_pred, average="macro"), "macro F1": f1_score(y_true, y_pred, average="macro"),
               "MCC": matthews_corrcoef(y_true, y_pred), "ROC AUC (ovr)": roc_auc_score(y_true, y_proba, multi_class="ovr"),
               "log loss": log_loss(y_true, y_proba, labels=list(range(k)))}
    if plot:
        fig, axes = plt.subplots(1, 3, figsize=(15, 4.4), gridspec_kw={"width_ratios": [1.1, 1, 1]})
        plot_confusion(confusion_matrix(y_true, y_pred, labels=list(range(k))), list(labels), axes[0], f"Confusion matrix – {name}")
        cmap = plt.get_cmap("tab10")
        for c in range(k):
            fpr, tpr, _ = roc_curve((y_true == c).astype(int), y_proba[:, c])
            axes[1].plot(fpr, tpr, color=cmap(c), lw=1.6, label=f"{labels[c]}")
        axes[1].plot([0, 1], [0, 1], color=GRAY, ls="--"); axes[1].set_xlabel("false positive rate"); axes[1].set_ylabel("true positive rate")
        axes[1].set_title(f"ROC one-vs-rest (macro AUC = {metrics['ROC AUC (ovr)']:.3f})"); axes[1].legend(fontsize=7, ncol=2 if k > 5 else 1, loc="lower right")
        f1s = f1_score(y_true, y_pred, average=None)
        axes[2].bar(range(k), f1s, color=RED, width=0.65); axes[2].set_xticks(range(k)); axes[2].set_xticklabels(labels, rotation=45 if k > 4 else 0, ha="right" if k > 4 else "center")
        axes[2].set_ylim(0, 1.05); axes[2].set_ylabel("F1"); axes[2].set_title("F1 per class")
        plt.tight_layout(); plt.show()
    return pd.Series(metrics, name=name)
def annotate_heatmap(ax, mat, fmt="{:.2f}"):
    """Write the values on an imshow heat map with a readable text color."""
    mat = np.asarray(mat, dtype=float)
    lo, hi = np.nanmin(mat), np.nanmax(mat)
    for i in range(mat.shape[0]):
        for j in range(mat.shape[1]):
            v = mat[i, j]
            ax.text(j, i, "–" if np.isnan(v) else fmt.format(v), ha="center", va="center", color="white" if (not np.isnan(v) and v > (lo + hi) / 2) else DARK)

### **1. Dataset and Preparation**

The same problem and the same **70/30 stratified split** of the other notebooks of the series: **malignant ( 1 ) × benign ( 0 )** tumours, the 10 *mean* attributes of the Breast Cancer dataset.

In [4]:
#Load the Breast Cancer Dataset ( Positive Class = Malignant )
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split

data = load_breast_cancer(as_frame=True)
cols = [c for c in data.data.columns if c.startswith("mean ")]            # 10 "mean" attributes
X = data.data[cols]
y = (data.target == 0).astype(int)                                        # 1 = malignant, 0 = benign
CLASS_NAMES = ["benign", "malignant"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=RANDOM_STATE, stratify=y)
print("train:", X_train.shape, "| test:", X_test.shape, "| malignant in train:", f"{y_train.mean():.1%}")
X_train.head(3)

train: (398, 10) | test: (171, 10) | malignant in train: 37.2%


,mean radius,mean texture,mean perimeter,mean area,mean smoothness,mean compactness,mean concavity,mean concave points,mean symmetry,mean fractal dimension
78,20.18,23.97,143.70,1245.0,0.129,0.345,0.375,0.160,0.291,0.081
330,16.03,15.51,105.80,793.2,0.095,0.137,0.120,0.070,0.178,0.060
378,13.66,15.15,88.27,580.6,0.083,0.075,0.042,0.025,0.179,0.059


In [ ]:
#Dictionary Used to Collect the Test Metrics of This Notebook ( the Cells Below Add Their Results to It )
results = {}

### **2. Implementation**

* Rules with a support below 5% or above 95%, and rules that cover exactly the same cases as another one, are discarded
* The linear terms are **winsorised** ( 5%-95% ) and scaled to a standard deviation of 0.4, so that they are comparable with the rules, as in the original paper
* The regularisation strength is chosen by **cross-validation on the training set** ( maximising the ROC AUC )
* The **importance** of a term is `|weight| × standard deviation of the feature`

In [ ]:
#RuleFit: Rules from Every Node of a Gradient-Boosting Ensemble + Linear Terms + L1 Logistic Regression
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.linear_model import LogisticRegressionCV, LogisticRegression

class RuleFit:
    def __init__(self, n_estimators=60, max_depth=3, learning_rate=0.1, subsample=0.5, min_support=0.05, use_rules=True, use_linear=True, seed=RANDOM_STATE):
        self.n_estimators, self.max_depth, self.learning_rate, self.subsample = n_estimators, max_depth, learning_rate, subsample
        self.min_support, self.use_rules, self.use_linear, self.seed = min_support, use_rules, use_linear, seed

    @staticmethod
    def _mask(X, conds):
        m = np.ones(len(X), bool)
        for j, op, t in conds:
            m &= (X[:, j] <= t) if op == "<=" else (X[:, j] > t)
        return m

    def _design(self, X):
        cols = []
        if self.use_rules:
            cols.append(np.array([self._mask(X, r) for r in self.rules_], dtype=float).T if self.rules_ else np.zeros((len(X), 0)))
        if self.use_linear:
            cols.append(0.4 * (np.clip(X, self.lo_, self.hi_) - self.mean_) / self.std_)
        return np.hstack(cols)

    def fit(self, X, y):
        X = np.asarray(X, dtype=float)
        y = np.asarray(y)
        self.names_ = list(getattr(self, "feature_names", [f"x{j}" for j in range(X.shape[1])]))
        ensemble = GradientBoostingClassifier(n_estimators=self.n_estimators, max_depth=self.max_depth, learning_rate=self.learning_rate, subsample=self.subsample, random_state=self.seed).fit(X, y)
        candidates = {}
        for est in ensemble.estimators_[:, 0]:
            t = est.tree_
            def walk(node, conds):
                if conds:
                    candidates.setdefault(tuple(sorted(conds)), None)
                if t.children_left[node] != -1:
                    f, thr = t.feature[node], round(float(t.threshold[node]), 6)
                    walk(t.children_left[node], conds + [(int(f), "<=", thr)]); walk(t.children_right[node], conds + [(int(f), ">", thr)])
            walk(0, [])
        self.n_generated_ = len(candidates)
        seen, rules = set(), []
        for key in sorted(candidates, key=lambda k: (len(k), k)):                         # shorter rules first when two rules cover the same cases
            m = self._mask(X, key)
            if not (self.min_support <= m.mean() <= 1 - self.min_support):
                continue
            sig = m.tobytes()
            if sig in seen:
                continue
            seen.add(sig); rules.append(key)
        self.rules_ = rules
        self.lo_, self.hi_ = np.percentile(X, 5, axis=0), np.percentile(X, 95, axis=0)
        clipped = np.clip(X, self.lo_, self.hi_)
        self.mean_, self.std_ = clipped.mean(axis=0), clipped.std(axis=0) + 1e-12
        Z = self._design(X)
        # Attention! Recent scikit-learn versions announce that `penalty` will be replaced by `l1_ratios`: the call still works, and a FutureWarning may appear
        self.model_ = LogisticRegressionCV(Cs=np.logspace(-2.5, 0.5, 10), penalty="l1", solver="liblinear", cv=5, scoring="roc_auc", max_iter=2000, random_state=self.seed).fit(Z, y)
        self.coef_, self.feature_std_ = self.model_.coef_[0], Z.std(axis=0)
        self.terms_ = ([("rule", r) for r in self.rules_] if self.use_rules else []) + ([("linear", j) for j in range(X.shape[1])] if self.use_linear else [])
        return self

    def predict_proba(self, X):
        return self.model_.predict_proba(self._design(np.asarray(X, dtype=float)))

    def predict(self, X):
        return (self.predict_proba(X)[:, 1] > 0.5).astype(int)

    def n_rules(self):
        return int(((self.coef_ != 0) & np.array([kind == "rule" for kind, _ in self.terms_])).sum())

    def n_terms(self):
        return int((self.coef_ != 0).sum())

    def term_text(self, term):
        kind, v = term
        return " AND ".join(f"{self.names_[j]} {op} {t:.4g}" for j, op, t in v) if kind == "rule" else f"{self.names_[v]} ( linear )"

    def importance_table(self):
        rows = [{"term": self.term_text(t), "kind": t[0], "weight": w, "importance": abs(w) * s, "support": (self._mask_for(t))} for t, w, s in zip(self.terms_, self.coef_, self.feature_std_) if w != 0]
        return pd.DataFrame(rows).sort_values("importance", ascending=False).reset_index(drop=True)

    def _mask_for(self, term):
        return np.nan

In [ ]:
#Fit RuleFit on the Training Set and Show the Most Important Terms
RuleFit.feature_names = list(X_train.columns)
rf = RuleFit().fit(X_train, y_train)
proba_rf = rf.predict_proba(X_test)[:, 1]
pred_rf = rf.predict(X_test)
print(f"rules generated by the ensemble: {rf.n_generated_} | after the filters: {len(rf.rules_)} | terms with a non-zero weight: {rf.n_terms()} ( rules: {rf.n_rules()}, linear: {rf.n_terms() - rf.n_rules()} ) | chosen C: {rf.model_.C_[0]:.3g}")
results["RuleFit"] = report_binary(y_test, pred_rf, proba_rf, "RuleFit", CLASS_NAMES)
imp = rf.importance_table().drop(columns="support")
fig, ax = plt.subplots(figsize=(10, 4.4))
top = imp.head(10).iloc[::-1]
labels = top["term"].str.replace("mean ", "", regex=False).map(lambda t: t if len(t) <= 62 else t[:60] + "…")
ax.barh(labels, top["weight"], color=[RED if w > 0 else DARK for w in top["weight"]]); ax.axvline(0, color=GRAY, lw=1)
ax.set_xlabel("weight ( red: towards malignant, dark: towards benign )"); ax.set_title("The 10 most important terms")
fig.subplots_adjust(left=0.55, right=0.97, bottom=0.14, top=0.9); plt.show()
pd.set_option("display.max_colwidth", 100)
imp.head(10).round(3)

### **3. Quality Metrics Specific to RuleFit**

In [ ]:
#Ablation: Rules Only, Linear Terms Only, or Both ( Test Metrics and Number of Terms )
rows = []
for label, params in [("rules + linear terms ( RuleFit )", dict()), ("rules only", dict(use_linear=False)), ("linear terms only ( sparse logistic regression )", dict(use_rules=False))]:
    m = rf if label.startswith("rules + ") else RuleFit(**params).fit(X_train, y_train)
    p = m.predict_proba(X_test)[:, 1]
    rows.append({"model": label, "non-zero terms": m.n_terms(), "test accuracy": accuracy_score(y_test, (p > 0.5).astype(int)), "test ROC AUC": roc_auc_score(y_test, p), "test PR AUC": average_precision_score(y_test, p)})
pd.DataFrame(rows).round(3)

In [ ]:
#Regularisation Path: Number of Non-Zero Terms and Cross-Validated ROC AUC as a Function of C
Z_train = rf._design(X_train.to_numpy(dtype=float))
Cs = rf.model_.Cs_
cv_auc = rf.model_.scores_[1].mean(axis=0)
nnz = [int((LogisticRegression(penalty="l1", solver="liblinear", C=c, max_iter=2000).fit(Z_train, y_train).coef_ != 0).sum()) for c in Cs]
fig, ax = plt.subplots(figsize=(7.5, 3.8))
ax.semilogx(Cs, cv_auc, color=RED, marker="o", label="CV ROC AUC ( training set )"); ax.axvline(rf.model_.C_[0], color=DARK, ls=":"); ax.set_xlabel("C ( larger = less regularisation )"); ax.set_ylabel("CV ROC AUC")
ax2 = ax.twinx(); ax2.semilogx(Cs, nnz, color=DARK, marker="s", label="non-zero terms"); ax2.set_ylabel("non-zero terms"); ax2.grid(False)
ax.set_title("Sparsity vs accuracy ( dotted: chosen C )")
plt.tight_layout(); plt.show()
pd.DataFrame({"C": Cs, "CV ROC AUC": cv_auc, "non-zero terms": nnz}).round(4)

In [ ]:
#Stability: 5 Different Seeds of the Boosting ( Random Subsamples of the Training Set )
rows, top_attrs, top_rules = [], [], []
for seed in range(5):
    m = RuleFit(seed=seed).fit(X_train, y_train)
    p = m.predict_proba(X_test)[:, 1]
    imp_m = m.importance_table()
    top = [t for t, w, s_ in zip(m.terms_, m.coef_, m.feature_std_) if w != 0]
    order = np.argsort([-abs(w) * s_ for w, s_ in zip(m.coef_, m.feature_std_) if w != 0])[:10]
    top_terms = [top[i] for i in order]
    top_attrs.append({rf.names_[j] for kind, v in top_terms for j in ([c[0] for c in v] if kind == "rule" else [v])})
    top_rules.append({m.term_text(t) for t in top_terms})
    rows.append({"seed": seed, "rules generated": m.n_generated_, "non-zero terms": m.n_terms(), "attributes in the top 10 terms": len(top_attrs[-1]), "test accuracy": accuracy_score(y_test, (p > 0.5).astype(int)), "test ROC AUC": roc_auc_score(y_test, p)})
stab = pd.DataFrame(rows)
pairs = [(a, b_) for i, a in enumerate(top_attrs) for b_ in top_attrs[i + 1:]]
rule_pairs = [(a, b_) for i, a in enumerate(top_rules) for b_ in top_rules[i + 1:]]
print(f"similarity ( Jaccard ) between seeds of the attributes in the top 10 terms: {np.mean([len(a & b_) / len(a | b_) for a, b_ in pairs]):.2f} | of the exact text of the top 10 rules: {np.mean([len(a & b_) / len(a | b_) for a, b_ in rule_pairs]):.2f}")
stab.round(3)

### **4. Evaluation**

In [ ]:
#Reference Models on the Same Split ( Is the Technique Better than a Trivial or a Standard Model? )
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

references = {"Majority class": DummyClassifier(strategy="prior"),
              "Decision tree ( depth 3 )": DecisionTreeClassifier(max_depth=3, random_state=RANDOM_STATE),
              "Logistic regression": make_pipeline(StandardScaler(), LogisticRegression(max_iter=2000))}
reference_rows = {}
for ref_name, model in references.items():
    model.fit(X_train, y_train)
    reference_rows[ref_name] = report_binary(y_test, model.predict(X_test), model.predict_proba(X_test)[:, 1], ref_name, CLASS_NAMES, plot=False)
metric_cols = ["accuracy", "precision", "recall", "F1", "ROC AUC", "PR AUC"]
pd.concat([pd.DataFrame(results).T[metric_cols], pd.DataFrame(reference_rows).T[metric_cols]]).round(3)

In [ ]:
#Repeated Stratified CV on the Training Set ( 5 Folds x 3 ): RuleFit and the Reference Models
from sklearn.model_selection import RepeatedStratifiedKFold, cross_validate
cv_rep = RepeatedStratifiedKFold(n_splits=5, n_repeats=3, random_state=RANDOM_STATE)
accs, aucs, sizes = [], [], []
for tr_idx, te_idx in cv_rep.split(X_train, y_train):
    m = RuleFit().fit(X_train.iloc[tr_idx], y_train.iloc[tr_idx])
    p = m.predict_proba(X_train.iloc[te_idx])[:, 1]
    accs.append(((p > 0.5).astype(int) == y_train.iloc[te_idx].to_numpy()).mean()); aucs.append(roc_auc_score(y_train.iloc[te_idx], p)); sizes.append(m.n_terms())
cv_rows = {"RuleFit": {"accuracy ( mean )": np.mean(accs), "accuracy ( std )": np.std(accs), "ROC AUC ( mean )": np.mean(aucs), "ROC AUC ( std )": np.std(aucs)}}
for ref_name, model in references.items():
    res = cross_validate(model, X_train, y_train, cv=cv_rep, scoring=["accuracy", "roc_auc"])
    cv_rows[ref_name] = {"accuracy ( mean )": res["test_accuracy"].mean(), "accuracy ( std )": res["test_accuracy"].std(), "ROC AUC ( mean )": res["test_roc_auc"].mean(), "ROC AUC ( std )": res["test_roc_auc"].std()}
print(f"RuleFit non-zero terms per fold: {np.mean(sizes):.1f} ± {np.std(sizes):.1f}")
pd.DataFrame(cv_rows).T.round(3)

In [ ]:
#Uncertainty of the Test Metrics ( Bootstrap of the Test Cases, 1,000 Resamples )
def bootstrap_ci(y_true, pred, score, n_boot=1000, seed=RANDOM_STATE):
    rng_b = np.random.default_rng(seed)
    y_true, pred, score = np.asarray(y_true), np.asarray(pred), np.asarray(score)
    acc, auc = [], []
    for _ in range(n_boot):
        i = rng_b.integers(0, len(y_true), len(y_true))
        if len(np.unique(y_true[i])) < 2:
            continue
        acc.append((pred[i] == y_true[i]).mean())
        auc.append(roc_auc_score(y_true[i], score[i]))
    return pd.DataFrame({"accuracy": np.percentile(acc, [2.5, 50, 97.5]), "ROC AUC": np.percentile(auc, [2.5, 50, 97.5])}, index=["2.5%", "median", "97.5%"]).T

print("the test set has", len(y_test), "cases: differences smaller than these intervals are not reliable")
bootstrap_ci(y_test, pred_rf, proba_rf).round(3)

In [ ]:
#Calibration of the Predicted Probabilities ( Brier Score and Reliability Curve )
from sklearn.metrics import brier_score_loss
from sklearn.calibration import calibration_curve

proba_rf_ARRAY = np.asarray(proba_rf, dtype=float)
frac_pos, mean_pred = calibration_curve(y_test, proba_rf_ARRAY, n_bins=5, strategy="quantile")                       # bins with the same number of cases
print(f"Brier score: {brier_score_loss(y_test, proba_rf_ARRAY):.4f} | Brier score of a model that always predicts the training prevalence: {brier_score_loss(y_test, np.full(len(y_test), y_train.mean())):.4f} ( lower is better )")
fig, ax = plt.subplots(figsize=(4.6, 4.2))
ax.plot([0, 1], [0, 1], color=GRAY, ls="--", label="perfect calibration"); ax.plot(mean_pred, frac_pos, color=RED, marker="o", label="RuleFit")
ax.set_xlabel("mean predicted probability ( 5 bins of equal size )"); ax.set_ylabel("observed share of malignant"); ax.set_title("Reliability curve"); ax.legend(fontsize=8)
plt.tight_layout(); plt.show()

### **5. Takeaways**

* **A sum of readable rules:** each non-zero term is a rule with a weight; the **L1 penalty** keeps only a few of the hundreds of candidates ( sparsity, regularisation path )
* The **ablation** tells where the performance comes from. On this dataset the **sparse linear model is both the smallest ( 5 terms ) and the most accurate** of the three ( test accuracy 0.953 and ROC AUC 0.989 against 0.930 and 0.975 for the models with rules, which use about 45 terms ): when the relationship is close to linear, the rules add complexity without benefit. RuleFit pays off when there are **interactions and non-linearities**, and the ablation is how to find out
* The **selected rules are unstable:** with different seeds of the boosting the exact text of the 10 most important rules does not repeat ( similarity 0.00 ), while the attributes they use overlap strongly ( 0.86 ). Read the rules as *the attributes and regions that matter*, not as fixed thresholds
* **Cost of interpretability:** RuleFit is a *global* explanation of a sparse linear-in-rules model, not an explanation of the original ensemble; compare it with the reference models in the cross-validation table
* The cross-validation was done on the training set only, with the ensemble and the penalty refitted in each fold